In [ ]:
pip install pyspark

In [1]:
import os

os.environ.pop("SPARK_HOME", None)
os.environ.pop("PYSPARK_PYTHON", None)
os.environ.pop("PYSPARK_DRIVER_PYTHON", None)

In [2]:
import os

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@11"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [ ]:
# from pyspark.sql import SparkSession
# from pyspark.sql.functions import *


# spark = SparkSession.builder \
#     .appName("EmployeeProject") \
#     .config("spark.jars", "/Users/harshvardhan/Downloads/postgresql-42.7.3.jar") \
#     .master("local[*]") \
#     .getOrCreate()

# print("Spark started ✅")

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("OmniRopute") \
    .config("spark.driver.extraClassPath", "/Users/harshvardhan/Downloads/postgresql-42.7.3.jar") \
    .config("spark.executor.extraClassPath", "/Users/harshvardhan/Downloads/postgresql-42.7.3.jar") \
    .master("local[*]") \
    .getOrCreate()

print("Spark started ✅")

### Employee

In [ ]:
emp_df = spark.read.csv("/Users/harshvardhan/Downloads/bootcamp-project/bootcamp-project/bronze/employee_data/employee_data.csv",header=True,inferSchema=True)
total_columns = {"emp_id", "age", "name"}

no_of_columns = set(emp_df.columns)
print(total_columns-no_of_columns)

# removing duplicates and null values
emp_df = emp_df.select(col("emp_id"),col("age"),col("name"))
emp_df = emp_df.dropna().dropDuplicates()

# writing in gold
emp_df.write.mode("append").parquet("/Users/harshvardhan/Downloads/bootcamp-project/bootcamp-project/gold/employee_data")

# PostgreSQL connection properties
url = "jdbc:postgresql://localhost:5432/employee_db"

properties = {
    "user": "harshvardhan",
    "password": "",   # put your password
    "driver": "org.postgresql.Driver"
}
spark._jvm.org.postgresql.Driver
# emp_df.write.jdbc(
#     url=url,
#     table="employee",
#     mode="append",   
#     properties=properties
# )

import traceback

try:
    emp_df.write.jdbc(
        url=url,
        table="employee",
        mode="append",
        properties=properties
    )
except Exception as e:
    print(e)
    traceback.print_exc()
# df = spark.createDataFrame([(1, "test")], ["id", "name"])

# df.write.jdbc(
#     url="jdbc:postgresql://localhost:5432/employee_db",
#     table="test_table",
#     mode="overwrite",
#     properties={
#         "user": "postgres",
#         "password": "1234",
#         "driver": "org.postgresql.Driver"
#     }
# )

#write to postgres
# emp_df.write.mode("append")

# update log files

# emp_df.show()

In [ ]:
# import boto3
# from pyspark.sql import SparkSession
# from pyspark.sql.functions import col

# # Define S3 paths
# bucket_name = "poc-bootcamp-capstone-group3"
# bronze_prefix = "poc-bootcamp-bronze/employee_data/"
# gold_path = f"s3://{bucket_name}/poc-bootcamp-gold/employee_data_output/"
# processed_file_key = f"{bronze_prefix}processed_files.txt"

# # Expected columns
# expected_columns = {"emp_id", "age", "name"}

# # PostgreSQL connection details
# pg_url = "jdbc:postgresql://54.174.233.120:5432/postgres"
# pg_properties = {
#     "user": "postgres",
#     "password": "11223344",
#     "driver": "org.postgresql.Driver"
# }
# pg_table = "employee"


# # Initialize Spark and boto3
# spark = SparkSession.builder.appName("DailyEmployeeProcessing").getOrCreate()
# s3 = boto3.client('s3')

# try:
#     # Ensure processed files log exists
#     try:
#         obj = s3.get_object(Bucket=bucket_name, Key=processed_file_key)
#         processed_files = {line.split(',')[0]: float(line.split(',')[1]) for line in obj['Body'].read().decode('utf-8').splitlines() if line}
#     except s3.exceptions.NoSuchKey:
#         processed_files = {}

#     # List all CSV files in bronze
#     files = s3.list_objects_v2(Bucket=bucket_name, Prefix=bronze_prefix).get('Contents', [])
#     csv_files = [f for f in files if f['Key'].endswith('.csv')]

#     # Identify new/updated files
#     to_process = []
#     for file in csv_files:
#         key = file['Key']
#         last_modified = file['LastModified'].timestamp()
#         if processed_files.get(key) != last_modified:
#             to_process.append((key, last_modified))

#     if not to_process:
#         print("INFO: No new or updated files to process. Exiting gracefully.")
#     else:
#         for key, last_modified in to_process:
#             print(f"Processing file: {key}")
#             df = spark.read.option("header", True).option("inferSchema", True).csv(f"s3://{bucket_name}/{key}")

#             # Check if all expected columns are present
#             actual_columns = set(df.columns)
#             missing = expected_columns - actual_columns
#             if missing:
#                 raise ValueError(f"ERROR: Missing expected columns in {key}: {missing}")

#             # Clean and validate
#             cleaned_df = df.select(
#                 col("emp_id").cast("string"),
#                 col("age").cast("int"),
#                 col("name").cast("string")
#             ).dropna().dropDuplicates()

#             # Write cleaned data to gold path
#             cleaned_df.write.mode("append").parquet(gold_path)

#             #Write to Postgres
#             cleaned_df.write.mode("append").jdbc(url=pg_url, table=pg_table, properties=pg_properties)

#             # Update log
#             processed_files[key] = last_modified

#         # Write updated log back to S3
#         log_content = "\n".join([f"{k},{v}" for k, v in processed_files.items()])
#         s3.put_object(Bucket=bucket_name, Key=processed_file_key, Body=log_content)

# except Exception as e:
#     print(f"ERROR: Job failed with exception: {e}")

In [ ]:
emp_df.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/employee_db") \
    .option("dbtable", "employee") \
    .option("user", "harshvardhan") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

In [ ]:
# from pyspark.sql import SparkSession
# from pyspark.sql.functions import col
# import os

# spark = SparkSession.builder.appName("Employee_ETL").getOrCreate()

# # Paths
# base_path = "/Users/harshvardhan/Downloads/bootcamp-project/bootcamp-project"
# bronze_path = f"{base_path}/bronze/employee_data"
# gold_path = f"{base_path}/gold/employee_data"
# log_file = f"{bronze_path}/processed_files.txt"

# # Get all CSV files
# all_files = [f for f in os.listdir(bronze_path) if f.endswith(".csv")]

# # Read processed files log
# if os.path.exists(log_file):
#     with open(log_file, "r") as f:
#         processed_files = set(f.read().splitlines())
# else:
#     processed_files = set()

# # Find new files
# new_files = [f for f in all_files if f not in processed_files]

# if not new_files:
#     print("No new files to process")
# else:
#     print(f"Processing files: {new_files}")

#     file_paths = [f"{bronze_path}/{f}" for f in new_files]

#     emp_df = spark.read.csv(
#         file_paths,
#         header=True,
#         inferSchema=True
#     )

#     # Validate columns
#     required_cols = {"emp_id", "age", "name"}
#     missing_cols = required_cols - set(emp_df.columns)

#     if missing_cols:
#         raise ValueError(f"Missing columns: {missing_cols}")

#     # Clean data
#     emp_df = (
#         emp_df
#         .select("emp_id", "age", "name")
#         .dropna()
#         .dropDuplicates()
#     )

#     # Write to Gold
#     emp_df.write.mode("append").parquet(gold_path)

#     # Write to PostgreSQL
#     emp_df.write \
#         .format("jdbc") \
#         .option("url", "jdbc:postgresql://localhost:5432/employee_db") \
#         .option("dbtable", "dim_employee") \
#         .option("user", "harshvardhan") \
#         .option("driver", "org.postgresql.Driver") \
#         .mode("append") \
#         .save()

#     # Update log file
#     with open(log_file, "a") as f:
#         for file in new_files:
#             f.write(file + "\n")

#     print("Processing completed and log updated")

## Vehicle_registry Data

In [ ]:
# from pyspark.sql import SparkSession
# from pyspark.sql.functions import *
# import os

# spark = SparkSession.builder.appName("SimplePipeline").getOrCreate()

# # -----------------------
# # Paths
# # -----------------------
# bronze_path = "/Users/harshvardhan/Downloads/bootcamp-project/bootcamp-project/bronze/vehicle_registry/"
# silver_path = "/Users/harshvardhan/Downloads/bootcamp-project/silver/vehicle_registry_clean/"

# log_folder = bronze_path + "processed_files/"
# checkpoint_file = log_folder + "processed.txt"


# # -----------------------
# # Step 0: Create log folder if not exists
# # -----------------------
# if not os.path.exists(log_folder):
#     os.makedirs(log_folder)


# # -----------------------
# # Step 1: Read processed files
# # -----------------------
# if os.path.exists(checkpoint_file):
#     with open(checkpoint_file, "r") as f:
#         processed_files = set(f.read().splitlines())
# else:
#     processed_files = set()


# # -----------------------
# # Step 2: Get new files only
# # -----------------------
# all_files = [f for f in os.listdir(bronze_path) if f.endswith(".csv")]
# new_files = [f for f in all_files if f not in processed_files]

# print("New Files:", new_files)


# # -----------------------
# # Step 3: Process only new files
# # -----------------------
# for file in new_files:

#     df = spark.read.csv(bronze_path + file, header=True, inferSchema=True)

#     # Rename columns
#     df = df.toDF("vin", "model", "mfg_year", "fuel_type")

#     # Cleaning
#     df = df.withColumn("vin", trim(col("vin"))) \
#            .withColumn("model", trim(col("model"))) \
#            .withColumn("fuel_type", trim(col("fuel_type")))

#     # Remove invalid VIN
#     df = df.filter(col("vin").rlike("^[A-Z0-9]{8,17}$"))

#     # Fill nulls
#     df = df.fillna({
#         "model": "UNKNOWN",
#         "fuel_type": "UNKNOWN"
#     })

#     # Fix year
#     df = df.withColumn(
#         "mfg_year",
#         when((col("mfg_year") < 1990) | (col("mfg_year") > year(current_date())), None)
#         .otherwise(col("mfg_year"))
#     )

#     # Standardize fuel
#     df = df.withColumn("fuel_type", upper(col("fuel_type")))

#     # Remove duplicates
#     df = df.dropDuplicates(["vin"])

#     # Audit columns
#     df = df.withColumn("processed_file", lit(file)) \
#            .withColumn("processed_at", current_timestamp())

#     # Save to Silver
#     df.write.mode("append").parquet(silver_path)

#     # -----------------------
#     # Update log file
#     # -----------------------
#     with open(checkpoint_file, "a") as f:
#         f.write(file + "\n")

#     print(f"Processed: {file}")


# print("Done ✅")

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import os

spark = SparkSession.builder.appName("SimplePipeline").getOrCreate()

# Paths
bronze_path = "/Users/harshvardhan/Downloads/bootcamp-project/bootcamp-project/bronze/vehicle_registry/"
silver_path = "/Users/harshvardhan/Downloads/bootcamp-project/silver/vehicle_registry_clean/"

log_folder = bronze_path + "processed_files/"
checkpoint_file = log_folder + "processed.txt"

# Create log folder
os.makedirs(log_folder, exist_ok=True)

# Read processed files
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, "r") as f:
        processed_files = set(f.read().splitlines())
else:
    processed_files = set()

# Get new files
all_files = [f for f in os.listdir(bronze_path) if f.endswith(".csv")]
new_files = [f for f in all_files if f not in processed_files]

print("New Files:", new_files)

# 🚀 NEW: create accumulator DF
final_df = None

# Process files
for file in new_files:

    df = spark.read.csv(bronze_path + file, header=True, inferSchema=True)

    df = df.toDF("vin", "model", "mfg_year", "fuel_type")

    df = df.withColumn("vin", trim(col("vin"))) \
           .withColumn("model", trim(col("model"))) \
           .withColumn("fuel_type", trim(col("fuel_type")))

    df = df.filter(col("vin").rlike("^[A-Z0-9]{8,17}$"))

    df = df.fillna({
        "model": "UNKNOWN",
        "fuel_type": "UNKNOWN"
    })

    df = df.withColumn(
        "mfg_year",
        when((col("mfg_year") < 1990) | (col("mfg_year") > year(current_date())), None)
        .otherwise(col("mfg_year"))
    )

    df = df.withColumn("fuel_type", upper(col("fuel_type")))

    df = df.dropDuplicates(["vin"])

    df = df.withColumn("processed_file", lit(file)) \
           .withColumn("processed_at", current_timestamp())

    # 🔥 UNION ALL FILES
    if final_df is None:
        final_df = df
    else:
        final_df = final_df.union(df)

    # Update log
    with open(checkpoint_file, "a") as f:
        f.write(file + "\n")

    print(f"Processed: {file}")

# ✅ WRITE ONCE (important)
if final_df is not None:
    final_df.write.mode("append").parquet(silver_path)

print("Done ✅")

In [ ]:
final_df.select(col("processed_file")).distinct().show(truncate=False)

## New vehcial_registry Code

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import os
import sys

# -----------------------
# 🛠️ Spark Setup (The "Fixed" Way)
# -----------------------
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@21"
os.environ["SPARK_HOME"] = "/opt/homebrew/opt/apache-spark/libexec"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Add Java 17+ permission fix
os.environ["PYSPARK_SUBMIT_ARGS"] = '--driver-java-options "--add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED" pyspark-shell'

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("VehiclePipeline") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

# -----------------------
# Paths
# -----------------------
base_dir = "/Users/harshvardhan/Downloads/bootcamp-project/bootcamp-project"
bronze_path = os.path.join(base_dir, "bronze/vehicle_registry")
silver_path = os.path.join(base_dir, "silver/vehicle_registry_clean")

log_folder = os.path.join(bronze_path, "processed_files")
checkpoint_file = os.path.join(log_folder, "processed.txt")

os.makedirs(log_folder, exist_ok=True)

# -----------------------
# Get New Files
# -----------------------
processed_files = set()
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, "r") as f:
        processed_files = set(f.read().splitlines())

all_files = [f for f in os.listdir(bronze_path) if f.endswith(".csv")]
new_files = [f for f in all_files if f not in processed_files]

if not new_files:
    print("No new files to process ✅")
    # In a notebook, use return; in a script, use sys.exit()
    # exit() 
else:
    print(f"Processing {len(new_files)} new files: {new_files}")

    # -----------------------
    # 🚀 Read with Explicit Schema
    # -----------------------
    # Explicit schema prevents the "Unable to Infer Schema" crash
    csv_schema = StructType([
        StructField("vin", StringType(), True),
        StructField("model", StringType(), True),
        StructField("mfg_year", IntegerType(), True),
        StructField("fuel_type", StringType(), True)
    ])

    file_paths = [os.path.join(bronze_path, f) for f in new_files]
    
    # Read files
    df = spark.read.csv(file_paths, header=True, schema=csv_schema)

    # -----------------------
    # Transformations
    # -----------------------
    # 1. Clean Strings
    df = df.withColumn("vin", trim(col("vin"))) \
           .withColumn("model", trim(col("model"))) \
           .withColumn("fuel_type", trim(col("fuel_type")))

    # 2. Filter valid VINs (Regex)
    df = df.filter(col("vin").rlike("^[A-Z0-9]{8,17}$"))

    # 3. Handle Nulls
    df = df.fillna({"model": "UNKNOWN", "fuel_type": "UNKNOWN"})

    # 4. Year Validation
    current_yr = 2026 # Or: spark.range(1).select(year(current_date())).collect()[0][0]
    df = df.withColumn(
        "mfg_year",
        when((col("mfg_year") < 1990) | (col("mfg_year") > current_yr), None)
        .otherwise(col("mfg_year"))
    )

    # 5. Standardization
    df = df.withColumn("fuel_type", upper(col("fuel_type")))
    df = df.dropDuplicates(["vin"])

    # 6. Metadata
    # df = df.withColumn("processed_file", regexp_extract(input_file_name(), r'([^/]+$)', 1)) \
    #        .withColumn("processed_at", current_timestamp())

    # -----------------------
    # 🚀 Write to Silver (Parquet)
    # -----------------------
    df.write.mode("append") \
        .partitionBy("fuel_type") \
        .parquet(silver_path)

    # -----------------------
    # Update log
    # -----------------------
    with open(checkpoint_file, "a") as f:
        for file in new_files:
            f.write(file + "\n")

    print(f"Successfully processed {len(new_files)} files to {silver_path} ✅")


In [ ]:
df.show()


## vehicle_assignment

In [ ]:
# from pyspark.sql.window import Window
# from pyspark.sql.functions import *

# # -----------------------
# # 🧼 STEP 1: Remove duplicates
# # -----------------------
# df = df.dropDuplicates(["vin", "driver_id", "start_timestamp", "end_timestamp"])

# # -----------------------
# # 🧼 STEP 2: Remove zero-duration
# # -----------------------
# df = df.filter(col("start_timestamp") != col("end_timestamp"))

# # -----------------------
# # 🔥 STEP 3: Fix same timestamp conflict
# # -----------------------
# window_same = Window.partitionBy("vin", "start_timestamp") \
#                     .orderBy(col("end_timestamp").desc_nulls_last())

# df = df.withColumn("rank", row_number().over(window_same)) \
#        .filter(col("rank") == 1) \
#        .drop("rank")

# # -----------------------
# # 🔥 STEP 4: TEMP replace NULL (only for processing)
# # -----------------------
# df = df.withColumn(
#     "end_timestamp",
#     when(col("end_timestamp").isNull(), lit(9999999999))
#     .otherwise(col("end_timestamp"))
# )

# # -----------------------
# # 🔥 STEP 5: Fix overlaps
# # -----------------------
# window_spec = Window.partitionBy("vin").orderBy("start_timestamp")

# df = df.withColumn("next_start", lead("start_timestamp").over(window_spec))

# df = df.withColumn(
#     "end_timestamp",
#     when(
#         col("next_start").isNotNull() &
#         (col("end_timestamp") > col("next_start")),
#         col("next_start")
#     ).otherwise(col("end_timestamp"))
# ).drop("next_start")

# # -----------------------
# # 🔥 STEP 6: Convert BACK to NULL (IMPORTANT)
# # -----------------------
# df = df.withColumn(
#     "end_timestamp",
#     when(col("end_timestamp") == 9999999999, None)
#     .otherwise(col("end_timestamp"))
# )

# # -----------------------
# # ✅ FINAL OUTPUT (ONLY REQUIRED COLUMNS)
# # -----------------------
# final_df = df.select(
#     "vin",
#     "driver_id",
#     "start_timestamp",
#     "end_timestamp",
#     "daily_rate"
# )

# final_df.show(20, False)




In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import os
import sys

# -----------------------
# 🛠️ Spark Setup
# -----------------------
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@21"
os.environ["SPARK_HOME"] = "/opt/homebrew/opt/apache-spark/libexec"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["PYSPARK_SUBMIT_ARGS"] = '--driver-java-options "--add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED" pyspark-shell'

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("VehicleAssignmentPipeline") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

# -----------------------
# 📂 Paths
# -----------------------
base_dir = "/Users/harshvardhan/Downloads/bootcamp-project/bootcamp-project"

bronze_path = os.path.join(base_dir, "bronze/vehicle_assignment")
silver_path = os.path.join(base_dir, "silver/vehicle_assignment_clean")

log_folder = os.path.join(bronze_path, "processed_files")
checkpoint_file = os.path.join(log_folder, "processed.txt")

os.makedirs(log_folder, exist_ok=True)

# -----------------------
# 📌 Get New Files
# -----------------------
processed_files = set()
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, "r") as f:
        processed_files = set(f.read().splitlines())

all_files = [f for f in os.listdir(bronze_path) if f.endswith(".csv")]
new_files = [f for f in all_files if f not in processed_files]

if not new_files:
    print("No new files to process ✅")
else:
    print(f"Processing {len(new_files)} files: {new_files}")

    # -----------------------
    # 📥 Schema
    # -----------------------
    schema = StructType([
        StructField("vin", StringType(), True),
        StructField("driver_id", StringType(), True),
        StructField("start_timestamp", LongType(), True),
        StructField("end_timestamp", LongType(), True),
        StructField("daily_rate", DoubleType(), True),
        StructField("region", StringType(), True)
    ])

    file_paths = [os.path.join(bronze_path, f) for f in new_files]
    df = spark.read.csv(file_paths, header=True, schema=schema)

    # -----------------------
    # 🧼 CLEANING
    # -----------------------
    df = df.withColumn("vin", trim(col("vin"))) \
           .withColumn("driver_id", trim(col("driver_id"))) \
           .withColumn("region", trim(col("region")))

    df = df.filter(col("vin").rlike("^[A-Z0-9]{6,17}$"))

    # Remove duplicates
    df = df.dropDuplicates(["vin", "driver_id", "start_timestamp", "end_timestamp"])

    # Keep valid rows (important fix)
    df = df.filter(
        col("end_timestamp").isNull() |
        (col("start_timestamp") < col("end_timestamp"))
    )

    # -----------------------
    # 🔥 FIX SAME TIMESTAMP CONFLICT
    # -----------------------
    window_same = Window.partitionBy("vin", "start_timestamp") \
                        .orderBy(col("end_timestamp").desc_nulls_last())

    df = df.withColumn("rank", row_number().over(window_same)) \
           .filter(col("rank") == 1) \
           .drop("rank")

    # -----------------------
    # 🔥 HANDLE NULL END (TEMP)
    # -----------------------
    df = df.withColumn(
        "end_timestamp",
        when(col("end_timestamp").isNull(), lit(9999999999))
        .otherwise(col("end_timestamp"))
    )

    # -----------------------
    # 🔥 FIX OVERLAPS (FIXED BUG HERE)
    # -----------------------
    window_spec = Window.partitionBy("vin").orderBy("start_timestamp")

    df = df.withColumn("next_start", lead("start_timestamp").over(window_spec))

    df = df.withColumn(
        "end_timestamp",
        when(
            col("next_start").isNotNull() &
            (col("end_timestamp") != 9999999999) &   # 🔥 PROTECT ACTIVE
            (col("end_timestamp") > col("next_start")),
            col("next_start")
        ).otherwise(col("end_timestamp"))
    ).drop("next_start")

    # -----------------------
    # 🔥 ADD STATUS
    # -----------------------
    df = df.withColumn(
        "status",
        when(col("end_timestamp") == 9999999999, "ACTIVE")
        .otherwise("INACTIVE")
    )

    # -----------------------
    # 🔥 CONVERT BACK TO NULL
    # -----------------------
    df = df.withColumn(
        "end_timestamp",
        when(col("end_timestamp") == 9999999999, None)
        .otherwise(col("end_timestamp"))
    )

    # -----------------------
    # ✅ FINAL OUTPUT
    # -----------------------
    # final_df = df.select(
    #     "vin",
    #     "driver_id",
    #     "start_timestamp",
    #     "end_timestamp",
    #     "daily_rate",
    #     "status"
    # )


    # -----------------------
    # 🔥 CONVERT UNIX → DATE
    # -----------------------
    df = df.withColumn(
        "start_date",
        to_date(from_unixtime("start_timestamp"))
    )

    df = df.withColumn(
        "end_date",
        when(col("end_timestamp").isNotNull(),
            to_date(from_unixtime("end_timestamp")))
        .otherwise(None)
    )

    final_df = df.select(
    "vin",
    "driver_id",
    "start_date",
    "end_date",
    "daily_rate",
    "status"
)




    final_df.show(20, False)

    # -----------------------
    # 💾 WRITE
    # -----------------------
    final_df.write.mode("append") \
        .partitionBy("status") \
        .parquet(silver_path)

    # -----------------------
    # 📝 CHECKPOINT
    # -----------------------
    with open(checkpoint_file, "a") as f:
        for file in new_files:
            f.write(file + "\n")

    print(f"✅ Successfully processed {len(new_files)} files")

spark.stop()

Processing 1 files: ['vehicle_assignment.csv']
+---------+---------+----------+----------+----------+--------+
|vin      |driver_id|start_date|end_date  |daily_rate|status  |
+---------+---------+----------+----------+----------+--------+
|VIN000001|DRV_633  |2024-01-02|2024-01-03|584.75    |INACTIVE|
|VIN000001|DRV_633  |2024-01-03|2024-01-05|506.03    |INACTIVE|
|VIN000001|DRV_314  |2024-01-28|2024-01-29|641.87    |INACTIVE|
|VIN000001|DRV_314  |2024-01-29|2024-02-02|731.35    |INACTIVE|
|VIN000001|DRV_259  |2024-02-07|2024-02-08|500.42    |INACTIVE|
|VIN000001|DRV_314  |2024-03-05|NULL      |378.06    |ACTIVE  |
|VIN000001|DRV_618  |2024-04-04|2024-04-05|541.9     |INACTIVE|
|VIN000001|DRV_618  |2024-04-05|2024-04-07|699.77    |INACTIVE|
|VIN000001|DRV_719  |2024-04-11|2024-04-14|361.56    |INACTIVE|
|VIN000001|DRV_538  |2024-04-21|2024-04-24|308.75    |INACTIVE|
|VIN000001|DRV_173  |2024-04-30|2024-05-02|519.21    |INACTIVE|
|VIN000001|DRV_727  |2024-05-14|2024-05-19|429.02    |INA

✅ Successfully processed 1 files


## maintenance_schedules Code

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import os
import sys

# -----------------------
# 🛠️ Spark Setup
# -----------------------
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@21"
os.environ["SPARK_HOME"] = "/opt/homebrew/opt/apache-spark/libexec"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["PYSPARK_SUBMIT_ARGS"] = '--driver-java-options "--add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED" pyspark-shell'

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("MaintenancePipeline") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

# -----------------------
# 📂 Paths
# -----------------------
base_dir = "/Users/harshvardhan/Downloads/bootcamp-project/bootcamp-project"

bronze_path = os.path.join(base_dir, "bronze/maintenance_schedules")
silver_path = os.path.join(base_dir, "silver/maintenance_clean")

log_folder = os.path.join(bronze_path, "processed_files")
checkpoint_file = os.path.join(log_folder, "processed.txt")

os.makedirs(log_folder, exist_ok=True)

# -----------------------
# 📌 Get New Files
# -----------------------
processed_files = set()
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, "r") as f:
        processed_files = set(f.read().splitlines())

all_files = [f for f in os.listdir(bronze_path) if f.endswith(".csv")]
new_files = [f for f in all_files if f not in processed_files]

if not new_files:
    print("No new files to process ✅")
else:
    print(f"Processing {len(new_files)} files: {new_files}")

    # -----------------------
    # 📥 Schema
    # -----------------------
    schema = StructType([
        StructField("vin", StringType(), True),
        StructField("service_date", StringType(), True),  # raw (string)
        StructField("service_type", StringType(), True)
    ])

    file_paths = [os.path.join(bronze_path, f) for f in new_files]
    df = spark.read.csv(file_paths, header=True, schema=schema)

    # -----------------------
    # 🧼 CLEANING
    # -----------------------
    df = df.withColumn("vin", trim(col("vin"))) \
           .withColumn("service_type", trim(col("service_type")))

    # Remove invalid VINs
    df = df.filter(col("vin").rlike("^[A-Z0-9]{6,17}$"))

    # Handle NULL / invalid dates
    df = df.withColumn(
        "service_date",
        when(col("service_date").rlike("^\d{4}-\d{2}-\d{2}$"), col("service_date"))
        .otherwise(None)
    )

    # Convert to proper date
    df = df.withColumn(
        "service_date",
        to_date(col("service_date"), "yyyy-MM-dd")
    )

    # Remove NULL service_date
    df = df.filter(col("service_date").isNotNull())

    # Remove duplicates
    df = df.dropDuplicates(["vin", "service_date", "service_type"])

    # -----------------------
    # 🔥 STANDARDIZE SERVICE TYPE
    # -----------------------
    df = df.withColumn(
        "service_type",
        when(col("service_type").isNull() | (col("service_type") == ""), "UNKNOWN")
        .otherwise(upper(col("service_type")))
    )

    # -----------------------
    # 🔥 ADD FLAGS (EDGE CASE HANDLING)
    # -----------------------
    df = df.withColumn(
        "is_future_service",
        when(col("service_date") > current_date(), lit(1)).otherwise(lit(0))
    )

    df = df.withColumn(
        "is_past_service",
        when(col("service_date") < current_date(), lit(1)).otherwise(lit(0))
    )

    # -----------------------
    # 🔥 WINDOW (Multiple services same day)
    # -----------------------
    window_spec = Window.partitionBy("vin", "service_date").orderBy(col("service_type"))

    df = df.withColumn("rank", row_number().over(window_spec)) \
           .filter(col("rank") == 1) \
           .drop("rank")

    # -----------------------
    # 🔥 ADD YEAR / MONTH (Partition Ready)
    # -----------------------
    df = df.withColumn("year", year("service_date")) \
           .withColumn("month", month("service_date"))

    # -----------------------
    # ✅ FINAL OUTPUT
    # -----------------------
    final_df = df.select(
        "vin",
        "service_date",
        "service_type",
        # "is_future_service",
        # "is_past_service",
        "year",
        "month"
    )

    final_df.show(20, False)

    # -----------------------
    # 💾 WRITE (Silver Layer)
    # -----------------------
    final_df.write.mode("append") \
        .partitionBy("year", "month") \
        .parquet(silver_path)

    # -----------------------
    # 📝 CHECKPOINT
    # -----------------------
    with open(checkpoint_file, "a") as f:
        for file in new_files:
            f.write(file + "\n")

    print(f"✅ Successfully processed {len(new_files)} files")

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/17 11:30:55 WARN Utils: Your hostname, Harshs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.1.225.119 instead (on interface en0)
26/04/17 11:30:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 11:30:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


No new files to process ✅


### fuel_transactions code

In [40]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys
import logging

# -----------------------
# 🔧 Logging
# -----------------------
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("FuelPipeline")

# -----------------------
# 🛠️ Environment Setup
# -----------------------
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@21"
os.environ["SPARK_HOME"] = "/opt/homebrew/opt/apache-spark/libexec"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYSPARK_SUBMIT_ARGS"] = '--driver-java-options "--add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED" pyspark-shell'

# -----------------------
# 🚀 Spark Session
# -----------------------
spark = SparkSession.builder \
    .appName("FuelTransactionsPipeline") \
    .master("local[*]") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# -----------------------
# 📁 Paths
# -----------------------
BASE_DIR = "/Users/harshvardhan/Downloads/bootcamp-project/bootcamp-project"

BRONZE_PATH = os.path.join(BASE_DIR, "bronze/fuel_transactions")
SILVER_PATH = os.path.join(BASE_DIR, "silver/fuel_transactions_clean")

LOG_FOLDER = os.path.join(BRONZE_PATH, "processed_files")
CHECKPOINT_FILE = os.path.join(LOG_FOLDER, "processed.txt")

os.makedirs(LOG_FOLDER, exist_ok=True)

# -----------------------
# 📌 Schema
# -----------------------
SCHEMA = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("vin", StringType(), True),
    StructField("fuel_liters", FloatType(), True),
    StructField("odometer_reading", FloatType(), True),
    StructField("timestamp", StringType(), True)
])

# -----------------------
# 📥 Get New Files
# -----------------------
def get_new_files():
    processed = set()

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            processed = set(f.read().splitlines())

    all_files = [f for f in os.listdir(BRONZE_PATH) if f.endswith(".csv")]
    new_files = [f for f in all_files if f not in processed]

    return new_files


# -----------------------
# 🔄 Transform
# -----------------------
def transform(df):
    
    # Clean + Standardize
    df = df.select(
        trim(col("transaction_id")).alias("transaction_id"),
        trim(col("vin")).alias("vin"),
        col("fuel_liters"),
        col("odometer_reading"),
        to_timestamp(col("timestamp")).alias("event_time")
    )

    # VIN validation
    df = df.filter(col("vin").rlike("^[A-Z0-9]{8,17}$"))

    # Remove invalid fuel values
    df = df.filter(col("fuel_liters") > 0)

    # Remove invalid odometer
    df = df.filter(col("odometer_reading") > 0)

    # Deduplication
    df = df.dropDuplicates(["transaction_id"])

    # Metadata
    df = df.withColumn("ingestion_time", current_timestamp()) \
           .withColumn("source_file", input_file_name())

    df.show()
    
    return df




# -----------------------
# 💾 Write
# -----------------------
def write_data(df):
    df.write \
        .mode("append") \
        .partitionBy("event_date") \
        .parquet(SILVER_PATH)


# -----------------------
# 📝 Update Checkpoint
# -----------------------
def update_checkpoint(files):
    with open(CHECKPOINT_FILE, "a") as f:
        for file in files:
            f.write(file + "\n")


# -----------------------
# 🚀 Pipeline
# -----------------------
def run_pipeline():
    new_files = get_new_files()

    if not new_files:
        logger.info("No new files found ✅")
        return

    logger.info(f"Processing {len(new_files)} files: {new_files}")

    file_paths = [os.path.join(BRONZE_PATH, f) for f in new_files]

    df = spark.read.csv(file_paths, header=True, schema=SCHEMA)

    df_clean = transform(df)

    # Add partition column
    df_clean = df_clean.withColumn("event_date", to_date(col("event_time")))

    write_data(df_clean)

    update_checkpoint(new_files)

    logger.info("Fuel pipeline completed successfully ✅")

# -----------------------
# ▶️ Run
# -----------------------
if __name__ == "__main__":
    run_pipeline()

INFO:FuelPipeline:Processing 1 files: ['fuel_transactions.csv']


+--------------+-----------------+-----------+----------------+-------------------+--------------------+-----------+
|transaction_id|              vin|fuel_liters|odometer_reading|         event_time|      ingestion_time|source_file|
+--------------+-----------------+-----------+----------------+-------------------+--------------------+-----------+
|    TXN_100003|JHMFA1653TR61MFW0|     122.93|        176135.3|2026-04-07 10:16:00|2026-04-16 20:56:...|           |
|    TXN_100006|3VWFE21CSTDKWJWJQ|     116.39|        193636.0|2026-04-05 01:16:00|2026-04-16 20:56:...|           |
|    TXN_100007|JHMFA165RXNBFR5K4|     102.69|        140546.7|2026-04-07 13:04:00|2026-04-16 20:56:...|           |
|    TXN_100008|1HGCM826XB6R0AVUX|     119.56|        155960.9|2026-04-03 13:38:00|2026-04-16 20:56:...|           |
|    TXN_100009|2FTRX18WTTAO8DOTZ|      55.67|         48613.3|2026-04-05 06:16:00|2026-04-16 20:56:...|           |
|    TXN_100019|5YJSA1CNCBTG0D4LC|      31.69|         94380.9|2

INFO:FuelPipeline:Fuel pipeline completed successfully ✅


Py4JJavaError: An error occurred while calling o1861.showString.
: org.apache.spark.SparkException: [INTERNAL_ERROR] The "head" action failed. You hit a bug in Spark or the Spark plugins you use. Please, report this bug to the corresponding communities or vendors, and provide the full stack trace. SQLSTATE: XX000
	at org.apache.spark.SparkException$.internalError(SparkException.scala:107)
	at org.apache.spark.sql.execution.QueryExecution$.toInternalError(QueryExecution.scala:706)
	at org.apache.spark.sql.execution.QueryExecution$.withInternalError(QueryExecution.scala:719)
	at org.apache.spark.sql.classic.Dataset.$anonfun$withAction$1(Dataset.scala:2263)
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withNewExecutionId0$8(SQLExecution.scala:177)
	at org.apache.spark.sql.execution.SQLExecution$.withSessionTagsApplied(SQLExecution.scala:285)
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withNewExecutionId0$7(SQLExecution.scala:139)
	at org.apache.spark.JobArtifactS

In [3]:
from pyspark.sql.session import SparkSession
from pyspark.sql.functions import *


spark = SparkSession.builder.appName("OmniRoute").getOrCreate()

veh_df = spark.read.csv("/Users/harshvardhan/Downloads/bootcamp-project/bootcamp-project/bronze/vehicle_registry/raw_files/vehicle_registry_1.csv",header=True,inferSchema=True)

veh_df.show()

Error: LinkageError occurred while loading main class org.apache.spark.launcher.Main
	java.lang.UnsupportedClassVersionError: org/apache/spark/launcher/Main has been compiled by a more recent version of the Java Runtime (class file version 61.0), this version of the Java Runtime only recognizes class file versions up to 55.0
/Users/harshvardhan/Downloads/bootcamp-project/venv_spark/lib/python3.11/site-packages/pyspark/bin/spark-class: line 97: CMD: bad array subscript
head: illegal line count -- -1


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.